<a href="https://colab.research.google.com/github/mkobycheva/recommendation-system/blob/lstm/notebooks/04_lstm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 04 - LSTM Model

## 1. Connecting to the repository and Google Drive

In [11]:
!git clone https://github.com/mkobycheva/recommendation-system.git 2>/dev/null \
  || (cd recommendation-system && git pull)

%cd recommendation-system
# !pip install -r requirements.txt -q

import sys
sys.path.insert(0, '.')

from google.colab import drive
drive.mount('/content/drive')

/content/recommendation-system/recommendation-system
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Check out the right branch

In [2]:
!git checkout lstm && git pull

Branch 'lstm' set up to track remote branch 'lstm' from 'origin'.
Switched to a new branch 'lstm'
Already up to date.


## 2. Prepare data

In [3]:
import pandas as pd

DATA_DIR = '/content/drive/MyDrive/recsys-data'

books_train = pd.read_csv(f'{DATA_DIR}/books_train.csv')
books_valid = pd.read_csv(f'{DATA_DIR}/books_valid.csv')
books_test = pd.read_csv(f'{DATA_DIR}/books_test.csv')

movies_train = pd.read_csv(f'{DATA_DIR}/movies_train.csv')
movies_valid = pd.read_csv(f'{DATA_DIR}/movies_valid.csv')
movies_test = pd.read_csv(f'{DATA_DIR}/movies_test.csv')

In [4]:
from src.data.build_matrix import add_domain_item_ids, build_implicit_als_matrix

books_train = add_domain_item_ids(books_train, 'books')
books_valid = add_domain_item_ids(books_valid, 'books')
books_test = add_domain_item_ids(books_test, 'books')

movies_train = add_domain_item_ids(movies_train, 'movies')
movies_valid = add_domain_item_ids(movies_valid, 'movies')
movies_test = add_domain_item_ids(movies_test, 'movies')

train_df = pd.concat([books_train, movies_train], ignore_index=True)
valid_df = pd.concat([books_valid, movies_valid], ignore_index=True)
test_df = pd.concat([books_test, movies_test], ignore_index=True)

train_df[['user_id', 'item_id', 'domain']].head()

,user_id,item_id,domain
0,AFSKPY37N3C43SOI5IEXEK5JSIYA,books::0061713244,books
1,AFSKPY37N3C43SOI5IEXEK5JSIYA,books::0871139634,books
2,AFSKPY37N3C43SOI5IEXEK5JSIYA,books::0385521685,books
3,AFSKPY37N3C43SOI5IEXEK5JSIYA,books::0848732634,books
4,AFSKPY37N3C43SOI5IEXEK5JSIYA,books::0151014981,books


In [5]:
from sklearn.preprocessing import LabelEncoder

all_combined_items = pd.concat([
    train_df['item_id'],
    valid_df['item_id'],
    test_df['item_id']
]).unique()

encoder = LabelEncoder()
encoder.fit(all_combined_items)

train_df['item_idx'] = encoder.transform(train_df['item_id']) + 1
valid_df['item_idx'] = encoder.transform(valid_df['item_id']) + 1
test_df['item_idx'] = encoder.transform(test_df['item_id']) + 1

NUM_ITEMS = len(encoder.classes_) + 1
print(f"Number of unique items for LSTM: {NUM_ITEMS}")

Number of unique items for LSTM: 608159


In [6]:
import numpy as np

def generate_sequences(df, max_len=10):
    X, y = [], []

    df_sorted = df.sort_values(by=['user_id', 'timestamp'])

    for user_id, group in df_sorted.groupby('user_id'):
        item_list = group['item_idx'].tolist()

        if len(item_list) < 2:
            continue

        seq = item_list[:-1]
        target = item_list[-1]

        if len(seq) >= max_len:
            seq = seq[-max_len:]
        else:
            seq = [0] * (max_len - len(seq)) + seq

        X.append(seq)
        y.append(target)

    return np.array(X), np.array(y)


print("Make orders for Train...")
X_train, y_train = generate_sequences(train_df, max_len=10)

valid_combined = pd.concat([train_df, valid_df])
print("Make orders for Validation...")
X_val, y_val = generate_sequences(valid_combined, max_len=10)

print(f"Train: X = {X_train.shape}, y = {y_train.shape}")
print(f"Valid: X = {X_val.shape}, y = {y_val.shape}")

Make orders for Train...
Make orders for Validation...
Train: X = (127188, 10), y = (127188,)
Valid: X = (127188, 10), y = (127188,)


In [7]:
import torch
from torch.utils.data import Dataset, DataLoader

class RecSysDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = RecSysDataset(X_train, y_train)
val_dataset = RecSysDataset(X_val, y_val)

BATCH_SIZE = 256
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

for sample_x, sample_y in train_loader:
    print(f"Batch inputs (X): {sample_x.shape}")
    print(f"Batch corect outputs (y): {sample_y.shape}")
    break

Batch inputs (X): torch.Size([256, 10])
Batch corect outputs (y): torch.Size([256])


## 3. Main part

In [8]:
import torch.nn as nn

class RecSysLSTM(nn.Module):
    def __init__(self, num_items, embedding_dim=64, hidden_dim=128):
        super(RecSysLSTM, self).__init__()

        self.item_embeddings = nn.Embedding(num_embeddings=num_items,
                                            embedding_dim=embedding_dim,
                                            padding_idx=0)

        self.lstm = nn.LSTM(input_size=embedding_dim,
                            hidden_size=hidden_dim,
                            batch_first=True)

        self.fc = nn.Linear(hidden_dim, num_items)

    def forward(self, x):

        embedded = self.item_embeddings(x)
        out, (hidden, cell) = self.lstm(embedded)
        last_step = out[:, -1, :]
        logits = self.fc(last_step)

        return logits

In [25]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = RecSysLSTM(num_items=NUM_ITEMS, embedding_dim=64, hidden_dim=64).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=0)

optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

In [26]:
EPOCHS = 15

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for x_batch, y_batch in train_loader:

        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        outputs = model(x_batch)

        loss = criterion(outputs, y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    model.eval()
    val_loss = 0
    with torch.no_grad():
        for x_batch, y_batch in val_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            outputs = model(x_batch)
            loss = criterion(outputs, y_batch)
            val_loss += loss.item()

    avg_val_loss = val_loss / len(val_loader)

    print(f"Epoch [{epoch+1}/{EPOCHS}] | Train Loss: {avg_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

Epoch [1/15] | Train Loss: 13.2831 | Val Loss: 12.9964
Epoch [2/15] | Train Loss: 11.6409 | Val Loss: 13.4348
Epoch [3/15] | Train Loss: 11.2746 | Val Loss: 13.8236
Epoch [4/15] | Train Loss: 11.1992 | Val Loss: 13.9952
Epoch [5/15] | Train Loss: 11.1578 | Val Loss: 14.1187
Epoch [6/15] | Train Loss: 11.1287 | Val Loss: 14.2759
Epoch [7/15] | Train Loss: 11.1047 | Val Loss: 14.4005
Epoch [8/15] | Train Loss: 11.0822 | Val Loss: 14.5774
Epoch [9/15] | Train Loss: 11.0590 | Val Loss: 14.6032
Epoch [10/15] | Train Loss: 11.0333 | Val Loss: 14.7187
Epoch [11/15] | Train Loss: 11.0042 | Val Loss: 14.8399
Epoch [12/15] | Train Loss: 10.9702 | Val Loss: 14.9421
Epoch [13/15] | Train Loss: 10.9309 | Val Loss: 15.0176
Epoch [14/15] | Train Loss: 10.8858 | Val Loss: 15.2379
Epoch [15/15] | Train Loss: 10.8355 | Val Loss: 15.3117


## 4. Evaluate model

In [27]:
user_history_dict = {}
train_sorted = train_df.sort_values(by=['user_id', 'timestamp'])

for user_id, group in train_sorted.groupby('user_id'):
    item_list = group['item_idx'].tolist()
    if len(item_list) >= 10:
        seq = item_list[-10:]
    else:
        seq = [0] * (10 - len(item_list)) + item_list
    user_history_dict[user_id] = seq

def recommend_for_users(eval_users, target_domain, k=10):
    model.eval()
    recommendations = {}

    mask = np.full(NUM_ITEMS, -np.inf)
    for idx, label in enumerate(encoder.classes_):
        if label.startswith(f"{target_domain}::"):
            mask[idx + 1] = 0.0

    mask_tensor = torch.tensor(mask, dtype=torch.float, device=device)

    batch_size = 512
    for i in range(0, len(eval_users), batch_size):
        batch_users = eval_users[i:i+batch_size]

        X_batch_list = [user_history_dict.get(u, [0] * 10) for u in batch_users]
        X_batch = torch.tensor(X_batch_list, dtype=torch.long, device=device)

        with torch.no_grad():
            logits = model(X_batch)
            masked_logits = logits + mask_tensor.unsqueeze(0)

            _, top_k_idx = torch.topk(masked_logits, k=k, dim=-1)
            top_k_idx = top_k_idx.cpu().numpy()

        for j, u in enumerate(batch_users):
            user_recs_idx = top_k_idx[j]
            user_recs_labels = []
            for idx in user_recs_idx:
                if idx > 0:
                    user_recs_labels.append(encoder.classes_[idx - 1])
                else:
                    user_recs_labels.append("unknown")
            recommendations[u] = user_recs_labels

    return recommendations

In [28]:
def relevant_items_by_user(split_df, target_domain):
    domain_rows = split_df[split_df['domain'] == target_domain]
    return domain_rows.groupby('user_id')['item_id'].agg(set).to_dict()

In [29]:
from src.evaluation.metrics import ndcg_at_k, recall_at_k

def evaluate_multi_domain(split_df, k=10):
    """
    Multi-domain: recommend items from BOTH domains simultaneously.
    Model is trained jointly on all domains (collective model).
    Evaluate each domain separately.
    """
    results = {}

    for target_domain in ['books', 'movies']:
        relevant = relevant_items_by_user(split_df, target_domain)
        eval_users = list(relevant.keys())

        print(f"Evaluating {target_domain}: {len(eval_users)} users")

        recs = recommend_for_users(eval_users, target_domain, k=k)

        ndcg_scores, recall_scores = [], []
        for user_id in eval_users:
            user_recs = recs.get(user_id, [])
            user_relevant = relevant.get(user_id, set())
            if not user_relevant:
                continue
            ndcg_scores.append(ndcg_at_k(user_recs, user_relevant, k))
            recall_scores.append(recall_at_k(user_recs, user_relevant, k))

        results[target_domain] = {
            'ndcg@10': round(np.mean(ndcg_scores), 4),
            'recall@10': round(np.mean(recall_scores), 4),
            'n_users': len(ndcg_scores),
        }

    return results


In [30]:
# Check on validation set
K = 10
valid_results = evaluate_multi_domain(valid_df, k=K)

for domain, metrics in valid_results.items():
    print(f"\n{domain}:")
    print(f"  NDCG@10:   {metrics['ndcg@10']}")
    print(f"  Recall@10: {metrics['recall@10']}")
    print(f"  Users:     {metrics['n_users']}")

Evaluating books: 127188 users
Evaluating movies: 127188 users

books:
  NDCG@10:   0.0026
  Recall@10: 0.0052
  Users:     127188

movies:
  NDCG@10:   0.0092
  Recall@10: 0.016
  Users:     127188
